# Entrenamiento y validación con Alpha Vantage

Voy a entrenar los modelos utilizando únicamente el conjunto de entrenamiento y los compararé con los datos de validación. La prueba final se mantendrá separada hasta elegir completamente el modelo.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_alpha_vantage_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Dimensiones:", datos.shape)

Dimensiones: (4980, 21)


In [2]:
# Variables que utilizarán los modelos
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]

# Por ahora solo utilizo entrenamiento y validación
entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

X_entrenamiento = entrenamiento[variables_predictoras]
y_entrenamiento = entrenamiento["Objetivo"].astype(int)

X_validacion = validacion[variables_predictoras]
y_validacion = validacion["Objetivo"].astype(int)

print("Entrenamiento:", X_entrenamiento.shape)
print("Validación:", X_validacion.shape)

print("\nDistribución del objetivo en entrenamiento:")
print(
    y_entrenamiento
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

Entrenamiento: (4052, 15)
Validación: (522, 15)

Distribución del objetivo en entrenamiento:
Objetivo
0    0.4958
1    0.5042
Name: proportion, dtype: float64


In [3]:
# Utilizo la misma función para evaluar todos los modelos
def calcular_metricas(
    nombre,
    particion,
    y_real,
    y_predicho,
    probabilidades=None
):
    resultado = {
        "Modelo": nombre,
        "Particion": particion,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": None
    }

    if probabilidades is not None:
        resultado["ROC_AUC"] = roc_auc_score(
            y_real,
            probabilidades
        )

    return resultado


resultados = []

## Modelo base: clase mayoritaria

Este modelo siempre predice la clase más frecuente del entrenamiento. Sirve como referencia mínima para comprobar si los demás modelos realmente aportan algo.

In [4]:
modelo_mayoria = DummyClassifier(
    strategy="most_frequent"
)

modelo_mayoria.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_mayoria_entrenamiento = modelo_mayoria.predict(
    X_entrenamiento
)

pred_mayoria_validacion = modelo_mayoria.predict(
    X_validacion
)

prob_mayoria_entrenamiento = modelo_mayoria.predict_proba(
    X_entrenamiento
)[:, 1]

prob_mayoria_validacion = modelo_mayoria.predict_proba(
    X_validacion
)[:, 1]

resultados.append(
    calcular_metricas(
        "Clase mayoritaria",
        "Entrenamiento",
        y_entrenamiento,
        pred_mayoria_entrenamiento,
        prob_mayoria_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Clase mayoritaria",
        "Validación",
        y_validacion,
        pred_mayoria_validacion,
        prob_mayoria_validacion
    )
)

## Modelo base de persistencia

Este modelo supone que la dirección de la jornada actual continuará en la jornada siguiente. Si el retorno actual es positivo predice una subida, y si no, predice una bajada.

In [5]:
pred_persistencia_entrenamiento = (
    entrenamiento["Retorno_diario"] > 0
).astype(int)

pred_persistencia_validacion = (
    validacion["Retorno_diario"] > 0
).astype(int)

resultados.append(
    calcular_metricas(
        "Persistencia",
        "Entrenamiento",
        y_entrenamiento,
        pred_persistencia_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Persistencia",
        "Validación",
        y_validacion,
        pred_persistencia_validacion
    )
)

## Regresión logística

Este será el primer modelo que utilizará conjuntamente todas las variables predictoras. El escalado se ajustará únicamente con entrenamiento para no utilizar información de validación.

In [6]:
modelo_logistico = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_logistica_entrenamiento = modelo_logistico.predict(
    X_entrenamiento
)

pred_logistica_validacion = modelo_logistico.predict(
    X_validacion
)

prob_logistica_entrenamiento = modelo_logistico.predict_proba(
    X_entrenamiento
)[:, 1]

prob_logistica_validacion = modelo_logistico.predict_proba(
    X_validacion
)[:, 1]

resultados.append(
    calcular_metricas(
        "Regresión logística",
        "Entrenamiento",
        y_entrenamiento,
        pred_logistica_entrenamiento,
        prob_logistica_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Regresión logística",
        "Validación",
        y_validacion,
        pred_logistica_validacion,
        prob_logistica_validacion
    )
)

In [7]:
resultados_iniciales = pd.DataFrame(
    resultados
)

columnas_metricas = [
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

resultados_iniciales[columnas_metricas] = (
    resultados_iniciales[columnas_metricas]
    .round(4)
)

display(
    resultados_iniciales.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  Balanced_accuracy  Precision  \
Modelo              Particion                                               
Clase mayoritaria   Entrenamiento    0.5042             0.5000     0.5042   
                    Validación       0.4904             0.5000     0.4904   
Persistencia        Entrenamiento    0.4798             0.4797     0.4841   
                    Validación       0.4962             0.4960     0.4864   
Regresión logística Entrenamiento    0.5225             0.5218     0.5231   
                    Validación       0.5345             0.5367     0.5202   

                                   Recall      F1  ROC_AUC  
Modelo              Particion                               
Clase mayoritaria   Entrenamiento  1.0000  0.6704   0.5000  
                    Validación     1.0000  0.6581   0.5000  
Persistencia        Entrenamiento  0.4841  0.4841      NaN  
                    Validación     0.4883  0.4873      NaN  
Regresión logística Entrenamiento  0.5996  0.5587   0.5363  
                    Validación     0.6523  0.5789   0.5420

## Comparación sin Posicion_cierre

Voy a repetir la regresión logística eliminando `Posicion_cierre` para comprobar si el rendimiento depende principalmente de esta variable.

In [8]:
# Quito Posicion_cierre de las variables
variables_sin_posicion = [
    variable
    for variable in variables_predictoras
    if variable != "Posicion_cierre"
]

X_entrenamiento_sin_posicion = entrenamiento[
    variables_sin_posicion
]

X_validacion_sin_posicion = validacion[
    variables_sin_posicion
]

# Entreno nuevamente el modelo sin Posicion_cierre
modelo_logistico_sin_posicion = Pipeline([
    (
        "escalador",
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico_sin_posicion.fit(
    X_entrenamiento_sin_posicion,
    y_entrenamiento
)

pred_sin_posicion_entrenamiento = (
    modelo_logistico_sin_posicion.predict(
        X_entrenamiento_sin_posicion
    )
)

pred_sin_posicion_validacion = (
    modelo_logistico_sin_posicion.predict(
        X_validacion_sin_posicion
    )
)

prob_sin_posicion_entrenamiento = (
    modelo_logistico_sin_posicion.predict_proba(
        X_entrenamiento_sin_posicion
    )[:, 1]
)

prob_sin_posicion_validacion = (
    modelo_logistico_sin_posicion.predict_proba(
        X_validacion_sin_posicion
    )[:, 1]
)

resultados.append(
    calcular_metricas(
        "Logística sin Posicion_cierre",
        "Entrenamiento",
        y_entrenamiento,
        pred_sin_posicion_entrenamiento,
        prob_sin_posicion_entrenamiento
    )
)

resultados.append(
    calcular_metricas(
        "Logística sin Posicion_cierre",
        "Validación",
        y_validacion,
        pred_sin_posicion_validacion,
        prob_sin_posicion_validacion
    )
)

In [9]:
comparacion_logistica = pd.DataFrame(
    resultados
)

comparacion_logistica = comparacion_logistica[
    comparacion_logistica["Modelo"].isin([
        "Regresión logística",
        "Logística sin Posicion_cierre"
    ])
].copy()

comparacion_logistica[columnas_metricas] = (
    comparacion_logistica[columnas_metricas]
    .round(4)
)

display(
    comparacion_logistica.set_index(
        ["Modelo", "Particion"]
    )
)

#Posicion_cierre no está generando el comportamiento artificial observado con Yahoo Finance
#Puede mantenerse como una variable normal en los siguientes modelos

Accuracy  Balanced_accuracy  \
Modelo                        Particion                                    
Regresión logística           Entrenamiento    0.5225             0.5218   
                              Validación       0.5345             0.5367   
Logística sin Posicion_cierre Entrenamiento    0.5242             0.5235   
                              Validación       0.5326             0.5347   

                                             Precision  Recall      F1  \
Modelo                        Particion                                  
Regresión logística           Entrenamiento     0.5231  0.5996  0.5587   
                              Validación        0.5202  0.6523  0.5789   
Logística sin Posicion_cierre Entrenamiento     0.5244  0.6040  0.5614   
                              Validación        0.5188  0.6484  0.5764   

                                             ROC_AUC  
Modelo                        Particion               
Regresión logística           Entrenamiento   0.5363  
                              Validación      0.5420  
Logística sin Posicion_cierre Entrenamiento   0.5362  
                              Validación      0.5429

## Interpretación de los coeficientes

Como las variables fueron estandarizadas, puedo comparar el tamaño de sus coeficientes. Un valor positivo aumenta la probabilidad estimada de subida, mientras que un valor negativo la reduce.

Los coeficientes no representan relaciones causales, pero permiten comprobar qué variables influyen más en las decisiones de la regresión logística.

In [10]:
# Recupero los coeficientes del modelo con todas las variables (quiero ver cual tuvo mas peso)
coeficientes_todas = pd.Series(
    modelo_logistico
    .named_steps["modelo"]
    .coef_[0],
    index=variables_predictoras
)

# Recupero los coeficientes del modelo sin Posicion_cierre
coeficientes_sin_posicion = pd.Series(
    modelo_logistico_sin_posicion
    .named_steps["modelo"]
    .coef_[0],
    index=variables_sin_posicion
)

# Comparo los coeficientes de los dos modelos
comparacion_coeficientes = pd.DataFrame({
    "Coeficiente_con_todas": coeficientes_todas,
    "Coeficiente_sin_Posicion_cierre": coeficientes_sin_posicion
})

# El valor absoluto permite ordenar por importancia sin importar el signo
comparacion_coeficientes["Importancia_absoluta"] = (
    comparacion_coeficientes[
        "Coeficiente_con_todas"
    ].abs()
)

comparacion_coeficientes = (
    comparacion_coeficientes
    .sort_values(
        "Importancia_absoluta",
        ascending=False
    )
)

display(
    comparacion_coeficientes.round(4)
)

# multiplicando el coeficiente por el valor estandarizado de la variable en esa jornada obtengo la contribucion
# Para cada jornada, el modelo calcula:intercepto + contribución de Retorno_diario + contribución de Cuerpo_vela.....
# Esa suma todavía no es la probabilidad. Primero es una puntuación interna
# La regresión logística transforma después esa puntuación en una probabilidad, si supera el 50%, el modelo predice 1 (ESO PARA CADA JORNADA) 

# En el entrenamiento obtengo los coeficientes de las varaibles
# Despues en la validacion utilizo esos coeficientes como constantes x valor de la variable (escalada) = contribucion de la variable en esa jornada



,Coeficiente_con_todas,Coeficiente_sin_Posicion_cierre,Importancia_absoluta
Cuerpo_vela,-0.3462,-0.3536,0.3462
Retorno_diario,0.2536,0.2539,0.2536
Distancia_MA5,0.1146,0.1149,0.1146
RSI_14,0.1111,0.1102,0.1111
Rango_diario,-0.0948,-0.0951,0.0948
Distancia_MA10,-0.0873,-0.0874,0.0873
Retorno_lag_1,-0.0619,-0.0620,0.0619
Retorno_lag_2,-0.0581,-0.0584,0.0581
Volatilidad_5,0.0427,0.0428,0.0427
Distancia_MA20,-0.0387,-0.0382,0.0387


## Estabilidad anual en validación

Voy a revisar los resultados de 2023 y 2024 por separado. Esto permite comprobar si el rendimiento general se mantiene durante ambos años o si depende solamente de un periodo concreto

In [11]:
# Obtengo la fecha de la jornada que intenta predecir cada fila
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_validacion = (
    fecha_objetivo
    .loc[validacion.index]
    .dt.year
)

# Convierto las predicciones en series para mantener las fechas
pred_logistica_validacion_serie = pd.Series(
    pred_logistica_validacion,
    index=validacion.index
)

prob_logistica_validacion_serie = pd.Series(
    prob_logistica_validacion,
    index=validacion.index
)

pred_sin_posicion_validacion_serie = pd.Series(
    pred_sin_posicion_validacion,
    index=validacion.index
)

prob_sin_posicion_validacion_serie = pd.Series(
    prob_sin_posicion_validacion,
    index=validacion.index
)

resultados_anuales = []

# Evalúo cada año de validación por separado
for anio in [2023, 2024]:
    mascara_anio = (
        anio_objetivo_validacion == anio
    )

    y_anio = y_validacion.loc[
        mascara_anio
    ]

    resultado_con_todas = calcular_metricas(
        "Regresión logística",
        str(anio),
        y_anio,
        pred_logistica_validacion_serie.loc[
            mascara_anio
        ],
        prob_logistica_validacion_serie.loc[
            mascara_anio
        ]
    )

    resultado_con_todas["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales.append(
        resultado_con_todas
    )

    resultado_sin_posicion = calcular_metricas(
        "Logística sin Posicion_cierre",
        str(anio),
        y_anio,
        pred_sin_posicion_validacion_serie.loc[
            mascara_anio
        ],
        prob_sin_posicion_validacion_serie.loc[
            mascara_anio
        ]
    )

    resultado_sin_posicion["Filas"] = (
        mascara_anio.sum()
    )

    resultados_anuales.append(
        resultado_sin_posicion
    )

tabla_anual = pd.DataFrame(
    resultados_anuales
)

columnas_tabla_anual = [
    "Modelo",
    "Particion",
    "Filas",
    "Accuracy",
    "Balanced_accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC"
]

tabla_anual = tabla_anual[
    columnas_tabla_anual
]

tabla_anual[columnas_metricas] = (
    tabla_anual[columnas_metricas]
    .round(4)
)

display(
    tabla_anual.set_index(
        ["Modelo", "Particion"]
    )
)

,,Filas,Accuracy,Balanced_accuracy,Precision,Recall,F1,ROC_AUC
Modelo,Particion,,,,,,,
Regresión logística,2023,260,0.5731,0.5710,0.5636,0.7045,0.6263,0.5911
Logística sin Posicion_cierre,2023,260,0.5731,0.5713,0.5652,0.6894,0.6212,0.5945
Regresión logística,2024,262,0.4962,0.5013,0.4744,0.5968,0.5286,0.4953
Logística sin Posicion_cierre,2024,262,0.4924,0.4981,0.4717,0.6048,0.5300,0.4956


## Conclusión de la regresión logística

La regresión logística obtuvo un resultado un poco mejor que los modelos base cuando se evaluó toda la validación. Sin embargo, al revisar cada año por separado, se vio que el comportamiento no fue estable.

En 2023 el modelo sí consiguió resultados por encima del azar, pero en 2024 volvió a quedar prácticamente alrededor del 50 %. También comprobé que eliminar Posicion_cierre casi no cambió las métricas, así que esta variable no está explicando el rendimiento del modelo con los datos de Alpha Vantage.

Por ahora voy a conservar la regresión logística porque es un modelo fácil de interpretar y me sirve como referencia para comparar después los resultados de Random Forest y XGBoost.

-----------------------------------------------------------------------------------------------------------------------------------------

## Ajuste temporal final de la regresión logística

Antes de seleccionar el modelo final voy a ajustar la regularización de la regresión logística utilizando únicamente divisiones temporales dentro del entrenamiento.

También compararé nuevamente la versión con todas las variables y la versión sin Posicion_cierre. La validación principal se utilizará después del ajuste y la prueba final continuará separada.

In [12]:
from sklearn.model_selection import (
    GridSearchCV,
    TimeSeriesSplit
)

# Creo divisiones internas respetando el orden temporal
validacion_temporal_logistica = TimeSeriesSplit(
    n_splits=5,
    gap=1
)

# Pruebo distintos niveles de regularización
parametros_logistica = {
    "modelo__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],
    "modelo__class_weight": [
        None,
        "balanced"
    ]
}


def crear_busqueda_logistica():
    modelo = Pipeline([
        (
            "escalador",
            StandardScaler()
        ),
        (
            "modelo",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ])

    return GridSearchCV(
        estimator=modelo,
        param_grid=parametros_logistica,
        scoring={
            "balanced_accuracy": "balanced_accuracy",
            "roc_auc": "roc_auc"
        },
        refit="balanced_accuracy",
        cv=validacion_temporal_logistica,
        n_jobs=-1,
        return_train_score=True
    )


# Ajusto la versión con todas las variables
busqueda_logistica_todas = (
    crear_busqueda_logistica()
)

busqueda_logistica_todas.fit(
    X_entrenamiento,
    y_entrenamiento
)

# Ajusto la versión sin Posicion_cierre
busqueda_logistica_sin_posicion = (
    crear_busqueda_logistica()
)

busqueda_logistica_sin_posicion.fit(
    X_entrenamiento_sin_posicion,
    y_entrenamiento
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'modelo__C': [0.01, 0.1, ...], 'modelo__class_weight': [None, 'balanced']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.","{'balanced_accuracy': 'balanced_accuracy', 'roc_auc': 'roc_auc'}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'balanced_accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 

In [13]:
def mostrar_resultado_busqueda(
    nombre,
    busqueda
):
    mejor_indice = busqueda.best_index_

    roc_auc_interno = (
        busqueda
        .cv_results_["mean_test_roc_auc"][
            mejor_indice
        ]
    )

    print(nombre)

    print(
        "Balanced accuracy interna:",
        round(busqueda.best_score_, 4)
    )

    print(
        "ROC-AUC interno:",
        round(roc_auc_interno, 4)
    )

    print(
        "Mejores parámetros:",
        busqueda.best_params_
    )

    print()


mostrar_resultado_busqueda(
    "Logística con todas las variables",
    busqueda_logistica_todas
)

mostrar_resultado_busqueda(
    "Logística sin Posicion_cierre",
    busqueda_logistica_sin_posicion
)

Logística con todas las variables
Balanced accuracy interna: 0.5062
ROC-AUC interno: 0.4994
Mejores parámetros: {'modelo__C': 0.01, 'modelo__class_weight': None}

Logística sin Posicion_cierre
Balanced accuracy interna: 0.5108
ROC-AUC interno: 0.5007
Mejores parámetros: {'modelo__C': 0.01, 'modelo__class_weight': None}



In [14]:
modelo_logistico_ajustado = (
    busqueda_logistica_todas
    .best_estimator_
)

modelo_logistico_ajustado_sin_posicion = (
    busqueda_logistica_sin_posicion
    .best_estimator_
)

modelos_logisticos_ajustados = [
    (
        "Logística ajustada",
        modelo_logistico_ajustado,
        X_entrenamiento,
        X_validacion
    ),
    (
        "Logística ajustada sin Posicion_cierre",
        modelo_logistico_ajustado_sin_posicion,
        X_entrenamiento_sin_posicion,
        X_validacion_sin_posicion
    )
]

resultados_logistica_ajustada = []
predicciones_validacion_ajustadas = {}
probabilidades_validacion_ajustadas = {}

for (
    nombre,
    modelo,
    X_entrenamiento_modelo,
    X_validacion_modelo
) in modelos_logisticos_ajustados:

    pred_entrenamiento = modelo.predict(
        X_entrenamiento_modelo
    )

    pred_validacion = modelo.predict(
        X_validacion_modelo
    )

    prob_entrenamiento = modelo.predict_proba(
        X_entrenamiento_modelo
    )[:, 1]

    prob_validacion = modelo.predict_proba(
        X_validacion_modelo
    )[:, 1]

    resultados_logistica_ajustada.append(
        calcular_metricas(
            nombre,
            "Entrenamiento",
            y_entrenamiento,
            pred_entrenamiento,
            prob_entrenamiento
        )
    )

    resultados_logistica_ajustada.append(
        calcular_metricas(
            nombre,
            "Validación",
            y_validacion,
            pred_validacion,
            prob_validacion
        )
    )

    predicciones_validacion_ajustadas[
        nombre
    ] = pd.Series(
        pred_validacion,
        index=validacion.index
    )

    probabilidades_validacion_ajustadas[
        nombre
    ] = pd.Series(
        prob_validacion,
        index=validacion.index
    )

tabla_logistica_ajustada = pd.DataFrame(
    resultados_logistica_ajustada
)

tabla_logistica_ajustada[
    columnas_metricas
] = (
    tabla_logistica_ajustada[
        columnas_metricas
    ]
    .round(4)
)

display(
    tabla_logistica_ajustada.set_index(
        ["Modelo", "Particion"]
    )
)

Accuracy  \
Modelo                                 Particion                 
Logística ajustada                     Entrenamiento    0.5215   
                                       Validación       0.5249   
Logística ajustada sin Posicion_cierre Entrenamiento    0.5234   
                                       Validación       0.5345   

                                                      Balanced_accuracy  \
Modelo                                 Particion                          
Logística ajustada                     Entrenamiento             0.5206   
                                       Validación                0.5274   
Logística ajustada sin Posicion_cierre Entrenamiento             0.5226   
                                       Validación                0.5372   

                                                      Precision  Recall  \
Modelo                                 Particion                          
Logística ajustada                     Entrenamiento     0.5214  0.6207   
                                       Validación        0.5122  0.6562   
Logística ajustada sin Posicion_cierre Entrenamiento     0.5228  0.6285   
                                       Validación        0.5194  0.6797   

                                                          F1  ROC_AUC  
Modelo                                 Particion                       
Logística ajustada                     Entrenamiento  0.5667   0.5351  
                                       Validación     0.5753   0.5313  
Logística ajustada sin Posicion_cierre Entrenamiento  0.5708   0.5350  
                                       Validación     0.5888   0.5351

## Estabilidad anual de la regresión ajustada

Además del resultado conjunto, voy a comprobar nuevamente 2023 y 2024 por separado. La configuración final no debe depender únicamente de uno de los dos años.

In [15]:
# Identifico el año de la jornada que se intenta predecir
fecha_objetivo = (
    datos.index
    .to_series()
    .shift(-1)
)

anio_objetivo_validacion = (
    fecha_objetivo
    .loc[validacion.index]
    .dt.year
)

resultados_anuales_ajustados = []

for nombre in predicciones_validacion_ajustadas:

    predicciones_modelo = (
        predicciones_validacion_ajustadas[
            nombre
        ]
    )

    probabilidades_modelo = (
        probabilidades_validacion_ajustadas[
            nombre
        ]
    )

    for anio in [2023, 2024]:
        mascara_anio = (
            anio_objetivo_validacion == anio
        )

        resultado_anio = calcular_metricas(
            nombre,
            str(anio),
            y_validacion.loc[
                mascara_anio
            ],
            predicciones_modelo.loc[
                mascara_anio
            ],
            probabilidades_modelo.loc[
                mascara_anio
            ]
        )

        resultado_anio["Filas"] = (
            mascara_anio.sum()
        )

        resultados_anuales_ajustados.append(
            resultado_anio
        )

tabla_anual_ajustada = pd.DataFrame(
    resultados_anuales_ajustados
)

tabla_anual_ajustada = tabla_anual_ajustada[
    [
        "Modelo",
        "Particion",
        "Filas",
        "Accuracy",
        "Balanced_accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC"
    ]
]

tabla_anual_ajustada[
    columnas_metricas
] = (
    tabla_anual_ajustada[
        columnas_metricas
    ]
    .round(4)
)

display(
    tabla_anual_ajustada.set_index(
        ["Modelo", "Particion"]
    )
)

Filas  Accuracy  \
Modelo                                 Particion                    
Logística ajustada                     2023         260    0.5577   
                                       2024         262    0.4924   
Logística ajustada sin Posicion_cierre 2023         260    0.5692   
                                       2024         262    0.5000   

                                                  Balanced_accuracy  \
Modelo                                 Particion                      
Logística ajustada                     2023                  0.5555   
                                       2024                  0.4985   
Logística ajustada sin Posicion_cierre 2023                  0.5671   
                                       2024                  0.5078   

                                                  Precision  Recall      F1  \
Modelo                                 Particion                              
Logística ajustada                     2023          0.5509  0.6970  0.6154   
                                       2024          0.4720  0.6129  0.5333   
Logística ajustada sin Posicion_cierre 2023          0.5602  0.7045  0.6242   
                                       2024          0.4793  0.6532  0.5529   

                                                  ROC_AUC  
Modelo                                 Particion           
Logística ajustada                     2023        0.5722  
                                       2024        0.4928  
Logística ajustada sin Posicion_cierre 2023        0.5781  
                                       2024        0.4939

## Selección de la regresión logística

Después del ajuste temporal, la mejor configuración fue una regresión logística con `C = 0.01` y sin aplicar un equilibrio artificial entre las clases.

La versión sin `Posicion_cierre` obtuvo el mejor balanced accuracy en validación y se comportó un poco mejor durante 2024. Además, utiliza una variable menos y evita depender de la característica que anteriormente produjo el resultado sospechoso con Yahoo Finance.

Aun así, los resultados siguen siendo bajos y no se mantienen de forma estable entre los años. Por eso, esta regresión se selecciona como el mejor modelo entre los evaluados hasta ahora, pero todavía no puede considerarse un modelo robusto.

Antes de abrir la prueba final, dejo fijada esta configuración:

* Regresión logística
* StandardScaler
* C = 0.01
* class_weight = None
* Umbral de clasificación de 0.50
* Todas las variables excepto Posicion_cierre
